# Agents, Tools, MCPs and All That: Part 2
---------

## 1. Introduction
-------------------

In [my last post](https://michael-harmon.com/posts/mcp1/) I went over how to create agents with [LangChain](https://www.langchain.com/) and give them access to tools through a local MCP using [FastMCP](https://gofastmcp.com/getting-started/welcome). In this post I want to go over how to deploy this MCP through a CI/CD pipeline using [Terraform](https://www.hashicorp.com/en/products/terraform), [GitHub Actions](https://github.com/features/actions) and [Google Cloud Run](https://cloud.google.com/run?hl=en). I'll show how to use the remote MCP server with [LangChain](https://www.langchain.com/) agent like I did previously, and also show how to use the server with agent that uses [Google AntiGravity](https://antigravity.google/). Setting up a remote MCP server is important since it will allow people (or agents) anywhere in the world use you the tools.

Let's get into it!

## 2. CI/CD With Terraform & GitHub Actions
-------------------

In a [prior post](https://michael-harmon.com/posts/chatbot2/) I covered how to use [GitHub Actions](https://github.com/features/actions) to create a [CI/CD](https://en.wikipedia.org/wiki/CI/CD) pipeline to deploy a containerized application to [Google Cloud Run](https://cloud.google.com/run?hl=en). Google Cloud Run is a great option for our MCP server because the server itself does not have any state. Instead each of the tools calls APIS or databases ([MongoDB](https://www.mongodb.com/) and [Redis](https://redis.io/)) and just requires access to them by passing my credentials into the [Docker container](https://www.docker.com). I deployed my [MCP server](https://github.com/mdh266/langchain-fastmcp/blob/main/mymcp/server.py) using a GitHub actions yaml file called [build-image.yml](https://github.com/mdh266/langchain-fastmcp/blob/main/.github/workflows/build-image.yml). This job builds a Docker image, pushes it to DockerHub, and then deploys it to Google Cloud Run. All my credentials (API Keys, Database usernames/passwords) are stored in [GitHub secrets](https://docs.github.com/en/actions/how-tos/write-workflows/choose-what-workflows-do/use-secrets) are [pushed to the container](https://github.com/mdh266/langchain-fastmcp/blob/main/.github/workflows/build-image.yml#L54) during deployment to Cloud Run. It is important to not we do not pass the secrets when [building the image](https://github.com/mdh266/langchain-fastmcp/blob/main/.github/workflows/build-image.yml#L34) as that would make our credentials accessible to anyone that pulls the image! I covered a lot of the details in this CI/CD pipeline during in a prior post. The one thing I will add is that I added [this line](https://github.com/mdh266/langchain-fastmcp/blob/main/.github/workflows/build-image.yml#L18) to the job to allow me to have "Run Workflow" on the GitHub Actions page so that I can trigger the build without pushing code changes.

Now the new [GitHub Action job](https://github.com/mdh266/langchain-fastmcp/blob/main/.github/workflows/terraform.yaml) that I will talk about is about the Terraform.

## 3. Using A Remote MCP Server With An Agent
-------------------

Model context protocol improves upon local tools by offering standard protocols for using tools on a remote server. Agents do not need direct access to data or source code the tool uses. In some ways, MCP is for agents as the way APIs are for regular code. The standard MCP framework is [FastMCP](https://gofastmcp.com/getting-started/welcome). I created a FastMCP server here with several tools I'll discuss. 

Now in order to make an LangChain agent have access to the tools supplied by an MCP server we need to use a specialized MCP client for LangChain called [LangChain MCP Adapter](https://reference.langchain.com/python/langchain-mcp-adapters). This can be imported and set to the address of the MCP server and set transport type:

In [ ]:
from langchain_mcp_adapters.client import MultiServerMCPClient  

lc_client = MultiServerMCPClient({"config": {"url": f"https://{mcp-url}/mcp", "transport": "http"}})

Now we can get the tools:

In [4]:
tool_list = await lc_client.get_tools()
tool_list

[StructuredTool(name='get_weather', description='Fetch current weather for *city* from OpenWeatherMap.\n\nThe function reads the API key from the ``OPEN_WEATHER_MAP_API_KEY``\nenvironment variable unless an explicit ``api_key`` argument is supplied.', args_schema={'properties': {'city': {'type': 'string', 'description': 'City name to query, e.g. ``"London"``.'}, 'api_key': {'anyOf': [{'type': 'string'}, {'type': 'null'}], 'default': None, 'description': 'Optional explicit API key; if omitted the environment variable\nis used.'}}, 'required': ['city'], 'type': 'object', 'additionalProperties': False}, metadata={'_meta': {'fastmcp': {'tags': []}}}, handle_tool_error=<function _handle_mcp_tool_error at 0x119f9bce0>, response_format='content_and_artifact', coroutine=<function convert_mcp_tool_to_langchain_tool.<locals>.call_tool at 0x119ffe980>),
 StructuredTool(name='convert_address_to_point', description='Convert a street address to a Shapely Point object.', args_schema={'properties': {'

Notice these are [LangChain tools](https://docs.langchain.com/oss/python/langchain/tools)! The adapter converted the FastMCP tool to a LangChain one:

Now let's give the agent access to these tools from the MCP server. This just like giving the agent any other tools, 

In [ ]:
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv())

from langchain_groq import ChatGroq
from langchain.agents import create_agent

model = ChatGroq(model="openai/gpt-oss-120b")

agent = create_agent(model=model, tools=tool_list)

result = await agent.ainvoke({
    'messages': [{'role': 'user', 
                 'content': 'What police precinct is 306 West 54th Street, manhattana, manhattan in? Get me their info.'
}]})

In [ ]:
print(result.get("messages")[-1].content)

**NYC Police Precinct:** **18th Precinct**  

| Detail | Information |
|--------|--------------|
| **Precinct Number** | 18 |
| **Borough** | Manhattan |
| **Address** | 306 West 54th Street, New York, NY 10019 |
| **Phone** | 212‑767‑8400 |
| **Website** | https://www1.nyc.gov/site/nypd/bureaus/patrol/precincts/precinct-18.page |

The 18th Precinct serves the Midtown West area of Manhattan, including the vicinity of 306 West 54th Street. If you need to report a non‑emergency incident, request a police report, or obtain other services, you can call the precinct directly at the phone number above or visit the precinct in person during its regular business hours. For emergencies, always dial **911**.


## 4. Using A Remote MCP Server With Antigravity
-----------------------------------

<!--  -->